# SwiftSQL Public Benchmark: 官方 BIRD Mini-Dev (SQLite/PG/MySQL) + 防幻觉拒答 竞品横评

> **完全独立的第三方权威基准评测（SwiftSQL 训练阶段从未见过 BIRD 数据库）**
> 推荐运行时：**Colab Pro L4 GPU (24GB)**，完整横评约 15–25 分钟（~1 积分），不占用你本地 Mac mini 磁盘。

**横评对象（可自由增删）**：
- **`SwiftSQL-3B`** (`xiain/SwiftSQL-3B-LoRA`) & **`SwiftSQL-3B + 1-Shot Repair`**
- **`Base-3B`** (`Qwen/Qwen2.5-Coder-3B-Instruct`)
- **`XiYanSQL-3B`** (`XGenerationLab/XiYanSQL-QwenCoder-3B-2504`)
- **`Arctic-R1-7B`** (`Snowflake/Arctic-Text2SQL-R1-7B`，可选开关，7B 强化学习旗舰，重点对比 EX vs 延迟/输出 Token 数 vs 多方言 vs 拒答！)

**3 大评测赛道**：
1. **BIRD Mini-Dev SQLite（官方真实 `.sqlite` 数据库执行准确率 EX + 输出 Token 数 + 单题耗时）**
2. **BIRD Mini-Dev PostgreSQL & MySQL（企业级真实大表跨方言语法通过率）**
3. **公平防幻觉拒答测试（Missing-Column Refusal F1，所有竞品均明确被提示缺列可拒答）**


In [ ]:
# 1. 拉取仓库与评测依赖
import os, sys, time, json, re, sqlite3, zipfile, urllib.request
from pathlib import Path
import torch

REPO_DIR = "/content/swift-sql"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/2422725977/swift-sql.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q --upgrade unsloth datasets sqlglot pyyaml tqdm
!pip install -q --no-deps -e {REPO_DIR}
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

if torch.cuda.is_available():
    print(f"GPU Ready: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/(1024**3):.1f} GB)")
else:
    raise RuntimeError("请先切换运行时为 L4 GPU！")


In [ ]:
# 2. 下载官方 BIRD Mini-Dev (2026) 题库、三方言官方 Schema 与真实 SQLite 数据库
from datasets import load_dataset
from swiftsql.dialect import drop_column_from_ddl, extract_referenced_columns, extract_schema_tables_and_columns

BENCH_DIR = Path("/content/bird_mini_dev_bench")
BENCH_DIR.mkdir(parents=True, exist_ok=True)

# 加载 HF 官方 birdsql/bird_mini_dev (SQLite / MySQL / PostgreSQL 各 500 题)
bird_ds = load_dataset("birdsql/bird_mini_dev")
print("Loaded BIRD Mini-Dev splits:", {k: len(v) for k, v in bird_ds.items()})

# 下载官方 pinned schemas (sqlite.json / mysql.json / postgresql.json)
SCHEMAS = {}
for dialect, fname in [("sqlite", "sqlite.json"), ("mysql", "mysql.json"), ("postgresql", "postgresql.json")]:
    local_p = BENCH_DIR / fname
    if not local_p.exists():
        url = f"https://huggingface.co/datasets/birdsql/bird_mini_dev/raw/main/baseline_inference/schemas/{fname}"
        urllib.request.urlretrieve(url, local_p)
    SCHEMAS[dialect] = json.loads(local_p.read_text(encoding="utf-8"))
print("Loaded official BIRD schemas for DBs:", list(SCHEMAS["sqlite"].keys()))

# 下载官方 BIRD Mini-Dev 真实 SQLite 数据库 (~800MB，Colab 机房约 10-20 秒下完)
DB_ZIP = BENCH_DIR / "minidev.zip"
DB_EXTRACT = BENCH_DIR / "dbs"
if not DB_EXTRACT.exists():
    print("Downloading official BIRD Mini-Dev SQLite databases (minidev.zip)...")
    !wget -q --show-progress -O {DB_ZIP} https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip
    with zipfile.ZipFile(DB_ZIP, "r") as zf:
        zf.extractall(DB_EXTRACT)

# 自动定位每个 db_id 对应的 .sqlite 文件路径
SQLITE_DB_PATHS = {p.stem: p for p in DB_EXTRACT.rglob("*.sqlite")}
print(f"Found {len(SQLITE_DB_PATHS)} official BIRD .sqlite databases:", sorted(SQLITE_DB_PATHS.keys())[:11])


In [ ]:
# 3. 定义官方 SQLite 执行器、公平竞品 SQL 提取器与 3 赛道题集构造
import random
from tqdm.auto import tqdm
from swiftsql.dialect import validate_sql
from swiftsql.evaluation import results_match
from swiftsql.prompts import build_generation_messages, build_repair_messages, parse_model_response

# 评测规模控制（设为 500 可跑全量 BIRD Mini-Dev；设为 150 约 12-18 分钟跑完全部 4 个模型）
BIRD_SQLITE_N = 200      # 赛道 1：官方 BIRD SQLite 真实执行题数（最大 500）
BIRD_DIALECT_N = 100     # 赛道 2：PG + MySQL 各取 N 题测多方言合法率
REFUSAL_N = 80           # 赛道 3：缺列拒答压力测试题数
MAX_SCHEMA_CHARS = 5500  # 过滤超长巨型 schema，统一截断保护所有 3B/7B 模型上下文公平

def execute_bird_sqlite(db_path: Path, sql: str, max_vm_steps: int = 2_000_000, max_rows: int = 5000):
    check = validate_sql(sql, dialect="sqlite", require_read_only=True)
    if not check.valid:
        return False, [], check.error
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    try:
        steps = 0
        def _prog():
            nonlocal steps
            steps += 2000
            return 1 if steps > max_vm_steps else 0
        conn.set_progress_handler(_prog, 2000)
        rows = conn.execute(sql).fetchmany(max_rows)
        return True, rows, ""
    except Exception as exc:
        return False, [], str(exc)
    finally:
        conn.close()

def extract_competitor_sql(raw_text: str):
    """Fairly extract SQL (or CANNOT_ANSWER refusal) from any competitor model, stripping <think> blocks."""
    text = (raw_text or "").strip()
    # Strip R1/CoT <think>...</think> blocks
    no_think = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()
    if not no_think:
        no_think = text
    if "CANNOT_ANSWER" in no_think.upper():
        return False, ""
    # Prefer fenced ```sql ... ```
    blocks = re.findall(r"```(?:sql|postgresql|mysql|sqlite)?\s*(.*?)\s*```", no_think, flags=re.DOTALL | re.IGNORECASE)
    if blocks:
        return True, blocks[-1].strip().rstrip(";") + ";"
    # Fallback: find SELECT / WITH statement
    m = re.search(r"((?:WITH|SELECT)\b.*)", no_think, flags=re.DOTALL | re.IGNORECASE)
    if m:
        sql = m.group(1).strip().split("\n\n")[0].strip()
        return True, sql
    return True, no_think

def build_competitor_messages(schema: str, question: str, dialect: str, allow_refusal: bool = False):
    sys_msg = (
        f"You are an expert {dialect} Text-to-SQL assistant. "
        f"Given the database schema and question, output a valid read-only {dialect} SELECT query inside a ```sql code block."
    )
    if allow_refusal:
        sys_msg += " Never invent columns or tables. If the schema does NOT contain the required columns to answer the question, reply ONLY with CANNOT_ANSWER."
    user_msg = f"Dialect: {dialect}\n\nDatabase Schema:\n{schema}\n\nQuestion:\n{question}"
    return [{"role": "system", "content": sys_msg}, {"role": "user", "content": user_msg}]

# 构造赛道 1（官方 BIRD SQLite 可执行题）
rng = random.Random(42)
sqlite_items = []
for row in bird_ds["mini_dev_sqlite"]:
    db_id = row["db_id"]
    if db_id not in SQLITE_DB_PATHS or db_id not in SCHEMAS["sqlite"]:
        continue
    schema = SCHEMAS["sqlite"][db_id][:MAX_SCHEMA_CHARS]
    q = row["question"]
    if row.get("evidence"):
        q = f"{q}\nHint: {row['evidence']}"
    sqlite_items.append({
        "question_id": row["question_id"],
        "db_id": db_id,
        "db_path": SQLITE_DB_PATHS[db_id],
        "schema": schema,
        "question": q,
        "gold_sql": row["SQL"],
        "difficulty": row.get("difficulty", "unknown"),
    })
rng.shuffle(sqlite_items)
sqlite_items = sqlite_items[:BIRD_SQLITE_N]

# 预计算每题的 Gold 执行结果（确保题库自身可执行）
verified_bird = []
for it in tqdm(sqlite_items, desc="Pre-computing official BIRD gold results"):
    ok, rows, _ = execute_bird_sqlite(it["db_path"], it["gold_sql"])
    if ok:
        it["gold_rows"] = rows
        verified_bird.append(it)
print(f"Track 1 ready: {len(verified_bird)} executable official BIRD Mini-Dev SQLite tasks.")

# 构造赛道 2（官方 BIRD PostgreSQL & MySQL 多方言题）
dialect_items = []
for split_name, d_name in [("mini_dev_pg", "postgresql"), ("mini_dev_mysql", "mysql")]:
    rows_d = list(bird_ds[split_name])
    rng.shuffle(rows_d)
    for row in rows_d[:BIRD_DIALECT_N]:
        db_id = row["db_id"]
        if db_id not in SCHEMAS[d_name]:
            continue
        q = row["question"] + (f"\nHint: {row['evidence']}" if row.get("evidence") else "")
        dialect_items.append({
            "dialect": d_name,
            "schema": SCHEMAS[d_name][db_id][:MAX_SCHEMA_CHARS],
            "question": q,
        })
print(f"Track 2 ready: {len(dialect_items)} official BIRD PostgreSQL + MySQL tasks.")

# 构造赛道 3（基于 Gretel test 独立留出集构建 50% 可答 + 50% 缺列不可答 公平拒答横评）
gretel_test = list(load_dataset("gretelai/synthetic_text_to_sql", split="test"))
rng.shuffle(gretel_test)
refusal_items = []
for row in gretel_test:
    if len(refusal_items) >= REFUSAL_N:
        break
    from swiftsql.dialect import extract_ddl_only
    ddl = extract_ddl_only(row["sql_context"])
    cols_map = extract_schema_tables_and_columns(ddl)
    ref_cols = extract_referenced_columns(row["sql"], dialect="sqlite")
    cands = [c for t, cs in cols_map.items() if len(cs) >= 2 for c in cs if c.lower() in ref_cols and not c.lower().endswith("id")]
    if not cands:
        continue
    if len(refusal_items) % 2 == 0:
        mutated = drop_column_from_ddl(ddl, rng.choice(cands))
        if not mutated:
            continue
        refusal_items.append({"schema": mutated, "question": row["sql_prompt"], "dialect": "postgresql", "answerable": False})
    else:
        refusal_items.append({"schema": ddl, "question": row["sql_prompt"], "dialect": "postgresql", "answerable": True})
print(f"Track 3 ready: {len(refusal_items)} balanced refusal stress tasks (50% unanswerable / 50% answerable).")


In [ ]:
# 4. 统一批量评测引擎 + 多模型同台横评
from unsloth import FastLanguageModel

INCLUDE_7B_COMPETITORS = True  # 设为 False 可只测 3B 梯队（~8分钟）；True 额外测 Snowflake/Arctic-Text2SQL-R1-7B

MODELS_TO_BENCHMARK = [
    {"name": "SwiftSQL-3B",  "repo": "xiain/SwiftSQL-3B-LoRA",                      "kind": "swiftsql",   "batch": 16, "max_tok": 256, "load_4bit": False},
    {"name": "Base-3B",      "repo": "Qwen/Qwen2.5-Coder-3B-Instruct",              "kind": "competitor", "batch": 16, "max_tok": 256, "load_4bit": False},
    {"name": "XiYanSQL-3B",  "repo": "XGenerationLab/XiYanSQL-QwenCoder-3B-2504",   "kind": "competitor", "batch": 16, "max_tok": 256, "load_4bit": False},
]
if INCLUDE_7B_COMPETITORS:
    MODELS_TO_BENCHMARK.append(
        {"name": "Arctic-R1-7B", "repo": "Snowflake/Arctic-Text2SQL-R1-7B",         "kind": "competitor", "batch": 8,  "max_tok": 768, "load_4bit": True}
    )

def batch_infer(model, tok, prompts, batch_size=16, max_new_tokens=256, desc="Infer"):
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    texts = [None] * len(prompts)
    tok_counts = [0] * len(prompts)

    def _run(sub):
        inp = tok([prompts[i] for i in sub], return_tensors="pt", padding=True, truncation=True, max_length=2560).to("cuda")
        in_len = inp["input_ids"].shape[1]
        outs = model.generate(
            **inp,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
        gen_ids = outs[:, in_len:]
        decoded = tok.batch_decode(gen_ids, skip_special_tokens=True)
        for idx, g_ids, txt in zip(sub, gen_ids, decoded):
            texts[idx] = txt.strip()
            tok_counts[idx] = int((g_ids != tok.pad_token_id).sum().item())

    with torch.inference_mode():
        for start in tqdm(range(0, len(order), batch_size), desc=desc):
            q = [order[start : start + batch_size]]
            while q:
                sub = q.pop(0)
                try:
                    _run(sub)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if len(sub) == 1:
                        raise
                    mid = len(sub) // 2
                    q.insert(0, sub[mid:])
                    q.insert(0, sub[:mid])
    return texts, tok_counts

LEADERBOARD = {}

for cfg in MODELS_TO_BENCHMARK:
    mname = cfg["name"]
    print(f"\n==================== Benchmarking: {mname} ({cfg['repo']}) ====================")
    model, tok = FastLanguageModel.from_pretrained(
        model_name=cfg["repo"],
        max_seq_length=3072,
        dtype=None,
        load_in_4bit=cfg["load_4bit"],
    )
    FastLanguageModel.for_inference(model)

    def _make_prompt(schema, question, dialect, allow_refusal=False):
        if cfg["kind"] == "swiftsql":
            msgs = build_generation_messages(schema=schema, question=question, dialect=dialect, mode="auto")
        else:
            msgs = build_competitor_messages(schema=schema, question=question, dialect=dialect, allow_refusal=allow_refusal)
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    # --- Track 1: BIRD Mini-Dev SQLite Execution Accuracy ---
    t1_prompts = [_make_prompt(it["schema"], it["question"], "sqlite") for it in verified_bird]
    t0 = time.perf_counter()
    t1_raw, t1_toks = batch_infer(model, tok, t1_prompts, batch_size=cfg["batch"], max_new_tokens=cfg["max_tok"], desc=f"{mname} BIRD-SQLite")
    t1_sec = time.perf_counter() - t0

    ex_ok, syn_ok = 0, 0
    bird_errors = []
    for idx, (it, raw) in enumerate(zip(verified_bird, t1_raw)):
        if cfg["kind"] == "swiftsql":
            p = parse_model_response(raw, default_dialect="sqlite")
            sql = p["sql"]
        else:
            _, sql = extract_competitor_sql(raw)
        v = validate_sql(sql, dialect="sqlite", require_read_only=True)
        if v.valid:
            syn_ok += 1
        ok, rows, err = execute_bird_sqlite(it["db_path"], sql)
        matched = ok and results_match(rows, it["gold_rows"], order_sensitive="ORDER BY" in it["gold_sql"].upper())
        if matched:
            ex_ok += 1
        elif not ok:
            bird_errors.append((idx, sql, err))

    # --- Track 1B: SwiftSQL + 1-Shot Execution Repair (simulates API gateway guardrail) ---
    repair_ex_ok = ex_ok
    if cfg["kind"] == "swiftsql" and bird_errors:
        rep_prompts = [
            tok.apply_chat_template(
                build_repair_messages(
                    schema=verified_bird[idx]["schema"],
                    question=verified_bird[idx]["question"],
                    broken_sql=bad_sql,
                    error_message=err,
                    dialect="sqlite",
                ),
                tokenize=False,
                add_generation_prompt=True,
            )
            for idx, bad_sql, err in bird_errors
        ]
        rep_raw, _ = batch_infer(model, tok, rep_prompts, batch_size=cfg["batch"], max_new_tokens=256, desc=f"{mname} 1-Shot Repair")
        for (idx, _, _), rtxt in zip(bird_errors, rep_raw):
            p = parse_model_response(rtxt, default_dialect="sqlite")
            ok, rows, _ = execute_bird_sqlite(verified_bird[idx]["db_path"], p["sql"])
            if ok and results_match(rows, verified_bird[idx]["gold_rows"], order_sensitive="ORDER BY" in verified_bird[idx]["gold_sql"].upper()):
                repair_ex_ok += 1

    # --- Track 2: Official BIRD PostgreSQL & MySQL Multi-Dialect Syntax ---
    t2_prompts = [_make_prompt(it["schema"], it["question"], it["dialect"]) for it in dialect_items]
    t2_raw, _ = batch_infer(model, tok, t2_prompts, batch_size=cfg["batch"], max_new_tokens=cfg["max_tok"], desc=f"{mname} BIRD-PG/MySQL")
    d_ok = 0
    for it, raw in zip(dialect_items, t2_raw):
        sql = parse_model_response(raw, default_dialect=it["dialect"])["sql"] if cfg["kind"] == "swiftsql" else extract_competitor_sql(raw)[1]
        if validate_sql(sql, dialect=it["dialect"], require_read_only=True).valid:
            d_ok += 1

    # --- Track 3: Fair Missing-Column Refusal F1 ---
    t3_prompts = [_make_prompt(it["schema"], it["question"], it["dialect"], allow_refusal=True) for it in refusal_items]
    t3_raw, _ = batch_infer(model, tok, t3_prompts, batch_size=cfg["batch"], max_new_tokens=cfg["max_tok"], desc=f"{mname} Refusal")
    tp = fp = fn = 0
    for it, raw in zip(refusal_items, t3_raw):
        pred_ans = parse_model_response(raw)["answerable"] if cfg["kind"] == "swiftsql" else extract_competitor_sql(raw)[0]
        if not it["answerable"]:
            if not pred_ans:
                tp += 1
            else:
                fn += 1
        else:
            if not pred_ans:
                fp += 1
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    ref_f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

    # --- Single-request latency probe (15 BIRD samples) ---
    lats = []
    with torch.inference_mode():
        for p in t1_prompts[:15]:
            inp = tok([p], return_tensors="pt", truncation=True, max_length=2560).to("cuda")
            ts = time.perf_counter()
            model.generate(**inp, max_new_tokens=cfg["max_tok"], do_sample=False, pad_token_id=tok.pad_token_id)
            lats.append((time.perf_counter() - ts) * 1000.0)

    LEADERBOARD[mname] = {
        "BIRD_SQLite_EX": round(ex_ok / len(verified_bird), 4),
        "BIRD_PG_MySQL_Syntax": round(d_ok / len(dialect_items), 4),
        "Refusal_F1": round(ref_f1, 4),
        "Avg_Output_Tokens": round(sum(t1_toks) / len(t1_toks), 1),
        "Single_Latency_ms": round(sum(lats) / len(lats), 1),
        "Throughput_QPS": round(len(verified_bird) / max(t1_sec, 1e-6), 2),
    }
    if cfg["kind"] == "swiftsql":
        LEADERBOARD[f"{mname}+Repair"] = {
            **LEADERBOARD[mname],
            "BIRD_SQLite_EX": round(repair_ex_ok / len(verified_bird), 4),
        }

    del model
    torch.cuda.empty_cache()


In [ ]:
# 5. 打印最终权威横评排行榜（可直接贴入 Hugging Face Model Card 与 GitHub README）
names = list(LEADERBOARD.keys())
metrics = ["BIRD_SQLite_EX", "BIRD_PG_MySQL_Syntax", "Refusal_F1", "Avg_Output_Tokens", "Single_Latency_ms", "Throughput_QPS"]
header = f"{'Benchmark Metric':22s} | " + " | ".join(f"{n:>17s}" for n in names)
print("=" * len(header))
print(header)
print("-" * len(header))
for m in metrics:
    print(f"{m:22s} | " + " | ".join(f"{str(LEADERBOARD[n][m]):>17s}" for n in names))
print("=" * len(header))


In [ ]:
# 6. 自动释放 Colab Pro GPU 配额
from google.colab import runtime

AUTO_UNASSIGN = True  # 如需保留结果调试可改为 False
if AUTO_UNASSIGN:
    print("Benchmark finished! Releasing GPU runtime to save compute units...")
    runtime.unassign()
